#  Import libraries

In [ ]:
# !pip install requirements.txt

In [25]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# 2. Data Acquisation

## 2.1. Scrape Data

In [27]:
# To store temporarly scarped data
data = []

for page in range(1, 51):
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    for book in soup.find_all("article", class_="product_pod"):
        title = book.find('h3').find('a')['title']
        price = book.find('p', class_='price_color').text.strip()

        rating_words = book.find('p', class_='star-rating')['class'][1]

        stock = book.find('p', class_='instock availability').text.strip()
        book_link = book.find('h3').find('a')['href']

        book_url = 'https://books.toscrape.com/catalogue/' + book_link.replace('../', '')

        # Visit each page
        book_response = requests.get(book_url)

        book_soup = BeautifulSoup(
            book_response.text,
            'html.parser'
        )

        breadcrumb = book_soup.find('ul', class_='breadcrumb')

        category = breadcrumb.find_all('li')[2].text.strip()

        # Store scraped data
        data.append({
            'Title': title,
            'Price': price,
            'Rating': rating_words,
            'Stock': stock,
            'Category': category
        })

    print(f"Page {page} completed")

    # Small delay between pages
    time.sleep(0.2)


Page 1 completed
Page 2 completed
Page 3 completed
Page 4 completed
Page 5 completed
Page 6 completed
Page 7 completed
Page 8 completed
Page 9 completed
Page 10 completed
Page 11 completed
Page 12 completed
Page 13 completed
Page 14 completed
Page 15 completed
Page 16 completed
Page 17 completed
Page 18 completed
Page 19 completed
Page 20 completed
Page 21 completed
Page 22 completed
Page 23 completed
Page 24 completed
Page 25 completed
Page 26 completed
Page 27 completed
Page 28 completed
Page 29 completed
Page 30 completed
Page 31 completed
Page 32 completed
Page 33 completed
Page 34 completed
Page 35 completed
Page 36 completed
Page 37 completed
Page 38 completed
Page 39 completed
Page 40 completed
Page 41 completed
Page 42 completed
Page 43 completed
Page 44 completed
Page 45 completed
Page 46 completed
Page 47 completed
Page 48 completed
Page 49 completed
Page 50 completed


## 2.2. Store collected data to a dataframe and csv

In [28]:
books_df = pd.DataFrame(data)

books_df.to_csv('books.csv', index=False)

### Inspect total books scraped and first 5 observations

In [29]:
print(f"Total books: {len(books_df)}")
books_df.head()

Total books: 1000


,Title,Price,Rating,Stock,Category
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction
2,Soumission,Â£50.10,One,In stock,Fiction
3,Sharp Objects,Â£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History


### Inspect the dataset information

In [30]:
books_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Title     1000 non-null   str  
 1   Price     1000 non-null   str  
 2   Rating    1000 non-null   str  
 3   Stock     1000 non-null   str  
 4   Category  1000 non-null   str  
dtypes: str(5)
memory usage: 39.2 KB


### Inspect the dataset shape (row, col)

In [31]:
books_df.shape

(1000, 5)

### Check for missing values

In [32]:
books_df.isna().sum()

Title       0
Price       0
Rating      0
Stock       0
Category    0
dtype: int64

### Check data types

In [33]:
books_df.duplicated().sum()

np.int64(0)

# 3. Data Cleaning
After eximining the dataset, we have no misssing values. 
We will only clean price, category, and feature engineer `rating` and `stock`

In [34]:
# Feature engineer rating_words to numeric values

rating_numeric = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}


books_df['Rating Numeric'] = books_df['Rating'].map(rating_numeric)

# Clean price
books_df['Price'] = (books_df['Price'].astype(str).str.replace(r'[^0-9]', '', regex=True).astype(float))

# Clean category
books_df['Category'] = (books_df['Category'].astype(str).str.strip())

# Clean stock
books_df['Stock'] = (books_df['Stock'].astype(str).str.replace('\n', '', regex=False).str.strip())

# Create numerical value for stock. 1 = In stock, 0 = Out of stock
books_df['In_Stock'] = books_df['Stock'].str.contains('In stock', case=False, na=False).astype(int)

clean_df = books_df.copy()
clean_df.to_csv('clean_books_df.csv', index=False)
clean_df.info()
clean_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Title           1000 non-null   str    
 1   Price           1000 non-null   float64
 2   Rating          1000 non-null   str    
 3   Stock           1000 non-null   str    
 4   Category        1000 non-null   str    
 5   Rating Numeric  1000 non-null   int64  
 6   In_Stock        1000 non-null   int64  
dtypes: float64(1), int64(2), str(4)
memory usage: 54.8 KB


,Title,Price,Rating,Stock,Category,Rating Numeric,In_Stock
0,A Light in the Attic,5177.0,Three,In stock,Poetry,3,1
1,Tipping the Velvet,5374.0,One,In stock,Historical Fiction,1,1
2,Soumission,5010.0,One,In stock,Fiction,1,1
3,Sharp Objects,4782.0,Four,In stock,Mystery,4,1
4,Sapiens: A Brief History of Humankind,5423.0,Five,In stock,History,5,1


In [35]:
# Number of books in each genre
clean_df['Category'].value_counts()

Category
Default               152
Nonfiction            110
Sequential Art         75
Add a comment          67
Fiction                65
Young Adult            54
Fantasy                48
Romance                35
Mystery                32
Food and Drink         30
Childrens              29
Historical Fiction     26
Poetry                 19
Classics               19
History                18
Horror                 17
Womens Fiction         17
Science Fiction        16
Science                14
Music                  13
Business               12
Travel                 11
Thriller               11
Philosophy             11
Humor                  10
Autobiography           9
Art                     8
Religion                7
Psychology              7
Spirituality            6
New Adult               6
Christian Fiction       6
Self Help               5
Biography               5
Sports and Games        5
Health                  4
Politics                3
Contemporary            3
Chr

In [36]:
# Number of different genres
clean_df['Category'].nunique()

50

# Data Visualization